# 第2回：Pythonを読み、Copilotと少し変える

**今日の問い：分からないコードを、どうやって小さく理解し、安全に書き換えるか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 変数・リスト・辞書・条件分岐・繰り返し・関数を読める
- 型ヒント・docstring・防御的な入力検査を備えた関数を書く
- assertによる小さなテストで、境界値と例外を先に固定する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 型ヒント：引数と戻り値の型を明示する注釈
- docstring：関数の目的と使い方を書く文字列
- 例外：処理を続けられない理由を伝える仕組み
- 単体テスト：関数の入出力を自動で確かめる小さなコード
- 純粋関数：同じ入力へ常に同じ出力を返し副作用のない関数

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 変数・リスト・辞書

化学実験の小さな記録をPythonの値として表します。


In [ ]:
sample_name = "CMP-0001"
temperatures = [60, 75, 90]
experiment = {"sample_id": sample_name, "solvent": "EtOH", "active": 1}
print(type(sample_name), sample_name)
print(type(temperatures), temperatures)
print(type(experiment), experiment)


## TRY：`for`と`if`を読む

実行前に、何行表示されるか予想します。


In [ ]:
for temperature in temperatures:
    label = "高温条件" if temperature >= 75 else "低温条件"
    print(temperature, label)


## 関数に型ヒントとdocstringを付ける

引数と戻り値の型、目的を明示すると、読み手（と生成AI）が誤解しにくくなります。


In [ ]:
def celsius_to_kelvin(celsius: float) -> float:
    "摂氏をケルビンへ変換する。"
    return celsius + 273.15

converted = [celsius_to_kelvin(value) for value in temperatures]
print(converted)
help(celsius_to_kelvin)


## TRY：エラーを省略せず読む

エラー名とメッセージの末尾に、原因へ近い情報があります。


In [ ]:
try:
    temperatures[10]
except Exception as error:
    print(type(error).__name__)
    print(error)


## CHANGE

`temperatures`へ温度を1つ追加し、表示と変換結果を確認します。

## ASK COPILOT

気になるセルを貼り、「各行の実行後に変数の型と中身がどうなるか表で説明して」と依頼します。提案は1つずつ試します。


## DEEP DIVE：テストで守る小さなユーティリティ

実行できることと、正しいことは別です。境界値・異常値を先に決め、防御的な関数を書きます。


In [ ]:
def celsius_to_kelvin_checked(celsius: float) -> float:
    "型と物理的な下限を検査してから摂氏をケルビンへ変換する。"
    if not isinstance(celsius, (int, float)):
        raise TypeError("温度は数値で入力してください")
    if celsius < -273.15:
        raise ValueError("絶対零度より低い値は指定できません")
    return celsius + 273.15

for value in [25, -273.15, -300, "25"]:
    try:
        print(value, "->", round(celsius_to_kelvin_checked(value), 2))
    except (TypeError, ValueError) as error:
        print(value, "->", type(error).__name__, error)


### assertで期待結果を先に固定する

IQR法で外れ値を除く関数を書き、正常・空・NaN混在の3ケースを先に書いてから検証します。


In [ ]:
import numpy as np

def drop_outliers_iqr(values: list[float], k: float = 1.5) -> list[float]:
    "IQR法で外れ値を除いた値のリストを返す。NaNは事前に除く。"
    clean = [v for v in values if v == v]  # NaN(v != v)を除外
    if not clean:
        return []
    q1, q3 = np.percentile(clean, [25, 75])
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    return [v for v in clean if low <= v <= high]

assert drop_outliers_iqr([10, 11, 12, 13, 1000]) == [10, 11, 12, 13]
assert drop_outliers_iqr([]) == []
assert drop_outliers_iqr([5, 5, float("nan")]) == [5, 5]
print("すべてのテストを通過しました")


## CHALLENGE：関数の性質を調べる

外れ値を除いた後もう一度適用すると、さらに減るでしょうか（冪等か）。四分位が動くため、必ずしも一致しません。


In [ ]:
rng = np.random.default_rng(0)
sample = rng.normal(50, 5, 200).tolist()
once = drop_outliers_iqr(sample)
twice = drop_outliers_iqr(once)
print("1回適用後の件数:", len(once))
print("2回目でさらに減った件数:", len(once) - len(twice))
print("2回目で変化なし(冪等):", once == twice)


## よくある誤り

- Notebookを途中から実行して変数がない
- Copilotの長い修正を一度に採用する
- エラー全文を読まずにセルを繰り返し実行する

## SELF-STUDY（任意・30〜60分）

- 収率のリストから外れ値をIQRで除く関数を、型ヒントとテスト付きで書く
- 正常値・空リスト・NaN混在の3ケースを、期待結果を先に書いてから検証する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 型ヒントとdocstringは何の役に立つか
2. assertは何を保証し、何を保証しないか
3. 生成AIのコードを何で確認するか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
